# Guided EDA — application mart (slice 1)

**Collaborative session cut** — load → drop high-null columns → numeric vs categorical → class imbalance.

Out of this notebook: WOE/IV, modeling, correlations beyond the balance check, competition extracts.

**Disclaimer:** public demo data only (not live bank BFSI). Prefer the built **application mart**; fall back to the book AMEX sample pickle if the mart is missing.

| Column | Meaning |
| --- | --- |
| `customer_ID` | Application / customer id (not a feature) |
| `target` | `1` = bad (default), `0` = good |

## 1. Load data

Resolution order:

1. `data/credit/application_mart/application_mart.parquet` (preferred — already cleaned to one row per `customer_ID`)
2. Else `credit/data/train_df_sample.pkl` (Git LFS book sample) — mark clearly in the printout if this fallback is used

If the pickle is a pointer/empty LFS object, run `git lfs pull --include="credit/data/*"` first.

In [ ]:
from pathlib import Path

import pandas as pd

ID_COL = "customer_ID"
TARGET = "target"
NULL_DROP_THRESHOLD = 0.80

REPO_ROOT_CANDIDATES = [Path("../.."), Path(".")]


def resolve_repo_root() -> Path:
    for root in REPO_ROOT_CANDIDATES:
        if (root / "credit" / "notebooks").exists() or (root / "pyproject.toml").exists():
            return root.resolve()
    return Path("../..").resolve()


ROOT = resolve_repo_root()
MART_PATH = ROOT / "data" / "credit" / "application_mart" / "application_mart.parquet"
SAMPLE_PKL = ROOT / "credit" / "data" / "train_df_sample.pkl"

if MART_PATH.exists():
    source_path = MART_PATH
    source_label = "application mart (preferred)"
    df = pd.read_parquet(source_path)
elif SAMPLE_PKL.exists() and SAMPLE_PKL.stat().st_size > 1_000_000:
    source_path = SAMPLE_PKL
    source_label = "book AMEX sample pickle (fallback — not the mart)"
    df = pd.read_pickle(source_path)
else:
    raise FileNotFoundError(
        "Neither application mart nor usable sample pickle found.\n"
        f"  mart: {MART_PATH}\n"
        f"  pkl:  {SAMPLE_PKL}\n"
        "Build the mart or run: git lfs pull --include=\"credit/data/*\""
    )

print(f"source: {source_label}")
print(f"path:   {source_path}")
print(f"shape:  {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"id col present: {ID_COL in df.columns}; target present: {TARGET in df.columns}")
df.head(3)

## 2. Drop columns with ≥80% nulls

High missingness columns add little signal for a first pass and inflate the feature list.
We compute the null rate per column, drop those at or above **80%**, and keep a cleaned frame `df_clean`.

Id and target are never dropped here (they are required keys).

In [ ]:
rows_before, cols_before = df.shape
null_rate = df.isna().mean().sort_values(ascending=False)

print("Null-rate summary (all columns):")
print(null_rate.describe().to_string())
print()
print("Top 15 null rates:")
print(null_rate.head(15).to_string())

drop_cols = [
    c
    for c in null_rate.index[null_rate >= NULL_DROP_THRESHOLD]
    if c not in {ID_COL, TARGET}
]
df_clean = df.drop(columns=drop_cols)
rows_after, cols_after = df_clean.shape

print()
print(f"Dropped {len(drop_cols)} columns with null rate ≥ {NULL_DROP_THRESHOLD:.0%}")
print("Dropped columns:")
for c in drop_cols:
    print(f"  - {c}: {null_rate[c]:.1%}")
print()
print(f"Shape before: {rows_before:,} × {cols_before:,}")
print(f"Shape after:  {rows_after:,} × {cols_after:,}")

## 3. Numeric vs categorical features

**Rule used here (simple dtype heuristic):**

- Exclude `customer_ID` and `target` from feature lists.
- **Numeric features:** remaining columns with a numeric dtype (`int*`, `float*`, `bool` treated as numeric).
- **Categorical features:** remaining non-numeric columns (`object`, `string`, `category`, etc.).

This is a first-cut inventory, not a final modeling schema. AMEX-style codes that are stored as floats still land in numeric here.

In [ ]:
feature_cols = [c for c in df_clean.columns if c not in {ID_COL, TARGET}]
numeric_features = (
    df_clean[feature_cols].select_dtypes(include="number").columns.tolist()
)
categorical_features = [c for c in feature_cols if c not in numeric_features]

print(f"Feature columns (excl. id/target): {len(feature_cols)}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print()
print("Numeric sample:", numeric_features[:12])
print("Categorical sample:", categorical_features[:12] if categorical_features else "(none)")
print()
print("Dtype value counts (feature columns only):")
print(df_clean[feature_cols].dtypes.value_counts().to_string())

## 4. Class imbalance (`target`)

`target = 1` → **bad** (default). `target = 0` → **good**.

Credit default datasets are usually **imbalanced** (far more goods than bads). We report counts, rates, and a simple bar chart so the imbalance is visible before any modeling.

In [ ]:
BAD, GOOD = 1, 0

counts = df_clean[TARGET].value_counts(dropna=False).sort_index()
n_good = int((df_clean[TARGET] == GOOD).sum())
n_bad = int((df_clean[TARGET] == BAD).sum())
n_total = n_good + n_bad
pct_good = n_good / n_total
pct_bad = n_bad / n_total

balance = pd.DataFrame(
    {
        "label": ["good (0)", "bad (1)"],
        "n": [n_good, n_bad],
        "pct": [pct_good, pct_bad],
    }
)
print(balance.to_string(index=False, formatters={"pct": "{:.2%}".format}))
print()
print(f"Imbalance ratio (good:bad) ≈ {n_good / max(n_bad, 1):.2f}:1")
print(f"Default / bad rate: {pct_bad:.2%}")

ax = counts.plot(kind="bar", title="target balance (0=good, 1=bad)", rot=0, figsize=(5, 3))
ax.set_xlabel("target")
ax.set_ylabel("count")

## Slice 1 summary (verified local run)

- **Source:** application mart (preferred) — `data/credit/application_mart/application_mart.parquet`
- **Shape before null drop:** 100,000 rows × 920 columns
- **Columns dropped (≥80% null):** 106
- **Shape after null drop:** 100,000 rows × 814 columns
- **Numeric features:** 810 | **Categorical features:** 2 (`D_63_last`, `D_64_last`)
- **Class balance:** good (0) = 73,933 (73.93%); bad (1) = 26,067 (26.07%); good:bad ≈ 2.84:1

The target is **imbalanced toward goods** (bad rate ≈ 26%, not near 50%). Treat default as the minority/positive class in later scoring work.

**Stop here for this session.** Next guided slices (not in this notebook): deeper missingness patterns, WOE/IV, scoring (#145).